In [ ]:
import sys, os
from pathlib import Path

os.environ['USE_CUPY'] = 'false'

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.utils.map import load_obstacles_config
from src.classes.mapping import LidarGridMapVec
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.belief_quantized.belief_mdp_n import BeliefMDP_n_Localization

# Output dir
OUTDIR = PROJECT_ROOT / 'notebooks' /  'localization' / 'outputs' /'tmat_visuals'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Figures will be saved to:', OUTDIR)


In [ ]:
def build_mdp(n=5, dt=1.0, max_a=5.0):
    all_obstacles, area = load_obstacles_config(environment='toy2')
    model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=dt, max_a=max_a)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
    mdp = BeliefMDP_n_Localization(n=n, motion_model=model, measurement_model=sensor, obstacles=all_obstacles, _map=grid_map)
    
    # Set known map for localization (uses obstacle segments directly)
    mdp.set_known_map(all_obstacles)
    
    return mdp


def position_marginal(T_col, X_n, n_side):
    pos = X_n[:, :2]
    xs = np.unique(pos[:, 0]); ys = np.unique(pos[:, 1])
    x_to_idx = {v: i for i, v in enumerate(sorted(xs)[:n_side])}
    y_to_idx = {v: i for i, v in enumerate(sorted(ys)[:n_side])}
    heat = np.zeros((n_side, n_side))
    for s, p in enumerate(T_col):
        x, y = pos[s]
        ix = x_to_idx.get(x); iy = y_to_idx.get(y)
        if ix is not None and iy is not None:
            heat[iy, ix] += p
    return heat


def plot_case(mdp: BeliefMDP_n_Localization, i_state: int, k_action: int, title: str):
    T = mdp.T_mat
    X = mdp.SQ.X_n
    u = mdp.AQ.U[k_action]
    dt = mdp.motion_model.dt

    # Get full state (4D for DoubleIntegrator)
    x0_full = X[i_state]  # (4,) for DoubleIntegrator: [px, py, vx, vy]
    x0_pos = x0_full[:2]  # Position only for visualization
    
    # Compute deterministic next state using motion model
    x1_full = mdp.motion_model.f_bar(x0_full, u)  # (4,)
    x1_pos = x1_full[:2]  # Next position
    x1_vel = x1_full[2:]  # Next velocity (for info)

    col = T[:, i_state, k_action]
    heat = position_marginal(col, X, n_side=mdp.n)

    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: Position space with initial state and expected displacement
    axs[0].scatter(X[:, 0], X[:, 1], s=5, c='lightgray', alpha=0.3, label='quantized states')
    # Draw arrow from initial to expected next position
    dx = x1_pos[0] - x0_pos[0]
    dy = x1_pos[1] - x0_pos[1]
    axs[0].arrow(x0_pos[0], x0_pos[1], dx, dy, head_width=0.3, head_length=0.2, 
                 fc='red', ec='red', lw=2, length_includes_head=True, label='expected displacement')
    axs[0].plot([x0_pos[0]], [x0_pos[1]], marker='*', color='k', ms=15, label='initial state')
    axs[0].set_title(f"State {i_state}, action {k_action}, |u|={np.linalg.norm(u):.3f}\n"
                     f"Expected: pos→[{x1_pos[0]:.2f}, {x1_pos[1]:.2f}], vel→[{x1_vel[0]:.2f}, {x1_vel[1]:.2f}]")
    axs[0].set_xlabel('x'); axs[0].set_ylabel('y')
    axs[0].set_aspect('equal', adjustable='box')
    axs[0].grid(True, ls=':', alpha=0.5)
    axs[0].legend(fontsize=8)

    # Right: Heatmap of transition probabilities (position-marginalized)
    im = axs[1].imshow(heat, origin='lower', cmap='viridis', aspect='auto')
    fig.colorbar(im, ax=axs[1], fraction=0.046, label='probability')
    axs[1].set_title("T_mat position-marginalized (over velocity)")
    axs[1].set_xlabel('x bin index'); axs[1].set_ylabel('y bin index')
    
    # Mark the initial position bin
    pos = X[:, :2]
    xs = np.unique(pos[:, 0]); ys = np.unique(pos[:, 1])
    x_to_idx = {v: i for i, v in enumerate(sorted(xs)[:mdp.n])}
    y_to_idx = {v: i for i, v in enumerate(sorted(ys)[:mdp.n])}
    ix0 = x_to_idx.get(x0_pos[0])
    iy0 = y_to_idx.get(x0_pos[1])
    if ix0 is not None and iy0 is not None:
        axs[1].plot(ix0, iy0, 'w*', markersize=15, markeredgecolor='black', markeredgewidth=1)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()

    # Display inline
    plt.show()

    # Save to disk as well
    outfile = OUTDIR / f"tmat_visual_{title.replace(' ', '_')}.png"
    fig.savefig(outfile, dpi=150, bbox_inches='tight')
    print('Saved figure to', outfile)


In [ ]:
# Build or load MDP (this will compute and cache T_mat on first run)
mdp = build_mdp(n=3, dt=1, max_a=2.0)
print('T_mat shape:', mdp.T_mat.shape)
print('State space size:', mdp.SQ.m_n, 'Action space size:', mdp.AQ.n_u)
print(f'Action space: square-lattice + circle mapping (n={mdp.AQ.n} quantization levels)')
print(f'Action magnitudes: min={np.min(np.linalg.norm(mdp.AQ.U, axis=1)):.3f}, max={np.max(np.linalg.norm(mdp.AQ.U, axis=1)):.3f}')
print(f'Known map obstacle segments: {len(mdp.known_map_obstacle_segments)} segments')
print()


In [ ]:
# Choose central state and sort actions by magnitude
i_state = mdp.SQ.m_n // 2
mags = np.linalg.norm(mdp.AQ.U, axis=1)
k_sorted = np.argsort(mags)

# Case 1: infeasible pair (if exists)
k_infeasible = None
for k in range(mdp.AQ.n_u):
    if not mdp.K_mask[i_state, k]:
        k_infeasible = k
        break
if k_infeasible is not None:
    plot_case(mdp, i_state, k_infeasible, title='infeasible_pair')
    assert np.allclose(mdp.T_mat[:, i_state, k_infeasible], 0.0)

# Case 2: small feasible control (stay in same pos bin)
k_small = None
for k in k_sorted:
    if mdp.K_mask[i_state, k]:
        k_small = k
        break
if k_small is not None:
    print(f"Small feasible control: action {k_small}, magnitude={np.linalg.norm(mdp.AQ.U[k_small]):.3f}")
    plot_case(mdp, i_state, k_small, title='small_control')
else:
    print(f"Warning: No small feasible action found for state {i_state}, skipping small control case")

# Case 3: largest feasible control
feasible_actions = [k for k in range(mdp.AQ.n_u) if mdp.K_mask[i_state, k]]
if len(feasible_actions) > 0:
    feasible_mags = [np.linalg.norm(mdp.AQ.U[k]) for k in feasible_actions]
    k_largest = feasible_actions[np.argmax(feasible_mags)]
    print(f"Largest feasible control: action {k_largest}, magnitude={np.linalg.norm(mdp.AQ.U[k_largest]):.3f}")
    plot_case(mdp, i_state, k_largest, title='largest_control')
else:
    print(f"Warning: No feasible actions found for state {i_state}, skipping largest control case")
